# LeetCode 656 — Coin Path bằng Python

- Link: https://leetcode.com/problems/coin-path/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Quy hoạch động (DP), Truy vết đường đi, Thứ tự từ điển (lexicographic), Hàng đợi đơn điệu (monotonic deque)

> **Ghi chú Premium:** đây là bài dành cho tài khoản LeetCode Premium. Phần đề bài dưới đây được viết lại theo hiểu biết về bài (chữ ký hàm, ví dụ, ràng buộc, định nghĩa thứ tự từ điển). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại** chi tiết đề trước khi nộp.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def cheapestJump(self, coins: List[int], maxJump: int) -> List[int]:

```

## 1. Giới thiệu & mục tiêu học

Thoạt nhìn đây là bài DP "nhảy trên mảng với chi phí nhỏ nhất" rất quen thuộc. Phần làm nó thành **Hard** là yêu cầu: khi có nhiều đường đi cùng chi phí, phải trả về đường **nhỏ nhất theo thứ tự từ điển**. Chi tiết này khiến cách DP "tự nhiên" (đi từ đầu tới cuối, nhớ ô trước) **sai**, và buộc ta phải nghĩ kỹ về **hướng** của DP.

Sau notebook này bạn sẽ:

1. Hiểu đề: chỉ số bắt đầu từ 1, ô `-1` không đáp được, chi phí gồm mọi ô đã đứng (kể cả ô đầu và ô cuối).
2. Viết brute force liệt kê mọi đường và so sánh theo cặp `(chi phí, đường đi)`.
3. Hiểu vì sao **DP ngược** (từ cuối về đầu) với quy tắc "hòa thì chọn bước nhảy tới ô nhỏ nhất" cho đúng đường nhỏ nhất theo từ điển — còn DP xuôi thì không.
4. Cài đặt DP ngược `O(n · maxJump)` và tối ưu `O(n)` bằng **deque đơn điệu** (min trên cửa sổ trượt).
5. Kiểm chứng mọi cách bằng brute force có seed, kèm phản ví dụ cho cách sai.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `coins` độ dài `n` (đánh số **từ 1** trong đề) và số nguyên `maxJump`.

- Bạn bắt đầu ở ô `1`. Từ ô `i` có thể nhảy tới bất kỳ ô `j` với `i + 1 <= j <= i + maxJump` và `j <= n`, **miễn là** `coins[j] != -1`.
- Mỗi lần **đứng** ở ô `i` (kể cả ô xuất phát và ô đích) bạn trả `coins[i]` xu.
- Tìm đường đi từ ô `1` tới ô `n` với **tổng chi phí nhỏ nhất**, trả về **danh sách chỉ số** (đánh số từ 1) các ô đã đứng. Nếu có nhiều đường cùng chi phí nhỏ nhất, trả về đường **nhỏ nhất theo thứ tự từ điển**. Nếu không tới được ô `n`, trả về `[]`.

**Thứ tự từ điển:** đường `P = [P1, ..., Pa]` nhỏ hơn `Q = [Q1, ..., Qb]` nếu tại vị trí đầu tiên `i` mà `Pi != Qi` thì `Pi < Qi`; nếu không có vị trí khác nhau thì đường ngắn hơn nhỏ hơn. (Vì mọi đường đều kết thúc ở `n` và chỉ số tăng dần, trường hợp "không có vị trí khác nhau" chỉ xảy ra khi hai đường trùng nhau.)

**Ví dụ 1**

```text
Input:  coins = [1, 2, 4, -1, 2], maxJump = 2
Output: [1, 3, 5]
Giải thích: các đường tới ô 5: 1->2->3->5 (1+2+4+2 = 9), 1->3->5 (1+4+2 = 7).
            Ô 4 là -1 nên không đáp được. Rẻ nhất là [1, 3, 5].
```

**Ví dụ 2**

```text
Input:  coins = [1, 2, 4, -1, 2], maxJump = 1
Output: []
Giải thích: mỗi lần chỉ nhảy 1 ô, buộc phải qua ô 4 (-1).
```

**Ràng buộc (theo hiểu biết về bài)**

- `1 <= coins.length <= 1000`
- `-1 <= coins[i] <= 100`
- `coins[1] != -1` (ô xuất phát luôn đứng được)
- `1 <= maxJump <= 100`

In [1]:
def path_cost(coins, path):
    """path: list chỉ số 1-based. Trả về chi phí, hoặc None nếu đường không hợp lệ."""
    if not path or path[0] != 1 or path[-1] != len(coins):
        return None
    total = 0
    for idx in path:
        if coins[idx - 1] == -1:
            return None
        total += coins[idx - 1]
    return total

ex1 = [1, 2, 4, -1, 2]
for p in ([1, 2, 3, 5], [1, 3, 5], [1, 3, 4, 5]):
    print(p, "->", path_cost(ex1, p))

[1, 2, 3, 5] -> 9
[1, 3, 5] -> 7
[1, 3, 4, 5] -> None


Hàm `path_cost` cũng kiểm tra đường có hợp lệ không (không đứng trên `-1`). Ta chưa kiểm tra bước nhảy `<= maxJump` — brute force ở mục 5 sẽ chỉ sinh đường hợp lệ.

## 3. Kiến thức nền

### 3.1. So sánh list theo thứ tự từ điển trong Python

Python so sánh list **đúng theo thứ tự từ điển**: phần tử đầu tiên khác nhau quyết định; nếu một list là tiền tố của list kia thì list ngắn hơn nhỏ hơn.

In [2]:
print([1, 2, 4, 5] < [1, 3, 5])      # True: vị trí 2: 2 < 3
print([1, 3, 5] < [1, 3, 4, 5])      # False: vị trí 3: 5 > 4
print([1, 2] < [1, 2, 3])            # True: tiền tố
print(min([[1, 3, 5], [1, 2, 4, 5], [1, 2, 3, 5]]))

True
False
True
[1, 2, 3, 5]


### 3.2. So sánh tuple `(chi phí, đường)` — hai tiêu chí trong một

`(cost, path)` so sánh `cost` trước, hòa thì so `path` theo từ điển — đúng luật của đề. Brute force chỉ cần `min` trên các tuple này.

In [3]:
cands = [(7, [1, 3, 5]), (9, [1, 2, 3, 5]), (7, [1, 2, 4, 5])]
print(min(cands))

(7, [1, 2, 4, 5])


### 3.3. Đổi chỉ số 1-based ↔ 0-based

Đề dùng chỉ số từ 1, Python dùng từ 0. Quy ước trong notebook: tính toán bằng chỉ số 0-based `i`, khi xuất kết quả thì cộng 1. Viết rõ ở một chỗ để tránh lệch.

In [4]:
path0 = [0, 2, 4]
print("0-based:", path0, "-> 1-based:", [i + 1 for i in path0])

0-based: [0, 2, 4] -> 1-based: [1, 3, 5]


### 3.4. Truy vết bằng mảng `nxt`

Thay vì lưu cả đường đi cho mỗi ô (tốn bộ nhớ `O(n^2)`), ta lưu `nxt[i]` = ô kế tiếp trên đường tốt nhất từ `i`. Dựng lại đường: bắt đầu từ ô đầu, đi theo `nxt` tới khi gặp ô cuối.

In [5]:
nxt = [2, -1, 4, -1, -1]          # 0 -> 2 -> 4 (ô 4 là đích)
path, i = [], 0
while i != -1:
    path.append(i + 1)
    i = nxt[i]
print(path)

[1, 3, 5]


### 3.5. Deque đơn điệu — min trên cửa sổ trượt

Muốn biết `min` của `k` phần tử gần nhất khi cửa sổ trượt, giữ một `deque` các chỉ số sao cho giá trị **đơn điệu**: phần tử mới vào thì loại các phần tử "bị trội" (giá trị không tốt hơn mà lại rời cửa sổ sớm hơn). Mỗi chỉ số vào/ra deque một lần → `O(n)` tổng.

In [6]:
from collections import deque

def sliding_min(a, k):
    dq, out = deque(), []
    for i, x in enumerate(a):
        while dq and a[dq[-1]] >= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(a[dq[0]])
    return out

print(sliding_min([4, 2, 12, 3, 8, 1, 7], 3))

[2, 2, 3, 1, 1]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Chi phí nhỏ nhất — DP quen thuộc

Đặt `dp[i]` = chi phí nhỏ nhất để đi từ ô `i` **tới đích** (tính cả `coins[i]` và `coins[n-1]`). Khi đó:

```text
dp[n-1] = coins[n-1]                          (nếu không phải -1)
dp[i]   = coins[i] + min(dp[j])  với j trong [i+1, i+maxJump], coins[j] != -1
dp[i]   = vô cùng nếu coins[i] == -1 hoặc không có j nào tới được đích
```

Đáp án chi phí: `dp[0]`.

### 4.2. Vì sao DP **ngược** hợp với thứ tự từ điển

So sánh hai đường **cùng xuất phát từ `i`**: phần tử đầu (`i`) giống nhau, nên phần tử **thứ hai** (ô nhảy tới đầu tiên) quyết định trước. Đường nào nhảy tới ô **nhỏ hơn** thì nhỏ hơn theo từ điển — **bất kể phần sau**. Nếu cùng ô nhảy tới `j`, thì so sánh phần còn lại, tức là so sánh hai đường xuất phát từ `j` — bài toán con!

Vậy với DP ngược: tại ô `i`, trong các `j` có `dp[j]` nhỏ nhất, **chọn `j` nhỏ nhất**. Đường tốt nhất từ `i` = `[i] + đường tốt nhất từ j`.

### 4.3. Tính tay ví dụ 1 (`maxJump = 2`, 0-based)

| i | coins[i] | các j hợp lệ | dp[j] | dp[i] | nxt[i] |
|---|---|---|---|---|---|
| 4 | 2 | — (đích) | — | 2 | -1 |
| 3 | -1 | — | — | ∞ | — |
| 2 | 4 | 3 (∞), 4 (2) | | 4 + 2 = 6 | 4 |
| 1 | 2 | 2 (6), 3 (∞) | | 2 + 6 = 8 | 2 |
| 0 | 1 | 1 (8), 2 (6) | | 1 + 6 = 7 | 2 |

Đường: `0 → 2 → 4` → 1-based `[1, 3, 5]`, chi phí 7 ✓.

### 4.4. Ví dụ hòa chi phí: `coins = [0, 0, 0, 0]`, `maxJump = 2`

Mọi đường đều chi phí 0: `[1,2,3,4]`, `[1,2,4]`, `[1,3,4]`. Nhỏ nhất theo từ điển là `[1, 2, 3, 4]` (vị trí 3: `3 < 4`). DP ngược: `nxt[2] = 3`, `nxt[1]`: `dp[2] = dp[3] = 0` → chọn `j = 2` nhỏ hơn; `nxt[0]`: `dp[1] = dp[2] = 0` → chọn `1`. Đường `0 → 1 → 2 → 3` ✓.

Lưu ý: đường **dài hơn** lại nhỏ hơn theo từ điển — trực giác "ít bước hơn thì tốt hơn" là **sai** ở đây.

In [7]:
def backward_table(coins, B):
    n = len(coins)
    INF = float('inf')
    dp, nxt = [INF] * n, [-1] * n
    if coins[-1] != -1:
        dp[-1] = coins[-1]
    print(f"{'i':>2} | {'coin':>4} | {'ứng viên (j: dp[j])':<28} | {'dp[i]':>5} | nxt")
    print(f"{n - 1:>2} | {coins[-1]:>4} | {'(đích)':<28} | {dp[-1]:>5} | -1")
    for i in range(n - 2, -1, -1):
        if coins[i] == -1:
            print(f"{i:>2} | {coins[i]:>4} | {'(không đứng được)':<28} | {'inf':>5} | -")
            continue
        cands = [(j, dp[j]) for j in range(i + 1, min(n, i + B + 1))]
        for j, d in cands:
            if coins[i] + d < dp[i]:
                dp[i], nxt[i] = coins[i] + d, j
        cs = ', '.join(f"{j}: {d}" for j, d in cands)
        print(f"{i:>2} | {coins[i]:>4} | {cs:<28} | {dp[i]:>5} | {nxt[i]}")
    return dp, nxt

backward_table(ex1, 2)
print()
backward_table([0, 0, 0, 0], 2)

 i | coin | ứng viên (j: dp[j])          | dp[i] | nxt
 4 |    2 | (đích)                       |     2 | -1
 3 |   -1 | (không đứng được)            |   inf | -
 2 |    4 | 3: inf, 4: 2                 |     6 | 4
 1 |    2 | 2: 6, 3: inf                 |     8 | 2
 0 |    1 | 1: 8, 2: 6                   |     7 | 2

 i | coin | ứng viên (j: dp[j])          | dp[i] | nxt
 3 |    0 | (đích)                       |     0 | -1
 2 |    0 | 3: 0                         |     0 | 3
 1 |    0 | 2: 0, 3: 0                   |     0 | 2
 0 |    0 | 1: 0, 2: 0                   |     0 | 1


([0, 0, 0, 0], [1, 2, 3, -1])

## 5. Cách 1 — Brute Force: liệt kê mọi đường

**Ý tưởng.** DFS từ ô 0, mỗi bước thử mọi ô nhảy tới hợp lệ. Khi tới ô cuối, so sánh `(chi phí, đường)` với đáp án tốt nhất hiện có.

In [8]:
def cheapest_brute(coins, B):
    n = len(coins)
    best = None
    path = [0]

    def dfs(i, cost):
        nonlocal best
        if i == n - 1:
            cand = (cost, [p + 1 for p in path])
            if best is None or cand < best:
                best = cand
            return
        for j in range(i + 1, min(n, i + B + 1)):
            if coins[j] != -1:
                path.append(j)
                dfs(j, cost + coins[j])
                path.pop()

    if coins[0] != -1:
        dfs(0, coins[0])
    return best[1] if best else []

print(cheapest_brute(ex1, 2), cheapest_brute(ex1, 1), cheapest_brute([0, 0, 0, 0], 2))
assert cheapest_brute(ex1, 2) == [1, 3, 5]
assert cheapest_brute(ex1, 1) == []
assert cheapest_brute([0, 0, 0, 0], 2) == [1, 2, 3, 4]

[1, 3, 5] [] [1, 2, 3, 4]


**Đọc code:**

- `path` là một list dùng chung, `append` khi đi xuống, `pop` khi quay lui (backtracking).
- Lưu `[p + 1 for p in path]` để kết quả là 1-based.
- `cand < best` so sánh tuple: chi phí trước, đường đi sau.

**Độ phức tạp.** Số đường đi có thể tới `~maxJump^(n/...)` — hàm mũ (với `maxJump = 2`, số đường cỡ Fibonacci `~1,6^n`). Mỗi đường tốn `O(n)` để sao chép. Space `O(n)` cho đệ quy.

**Nút thắt.** Các đường chia sẻ **hậu tố** chung (cùng đi qua ô `j` rồi tiếp tục giống nhau). Đường tốt nhất từ `j` chỉ cần tính **một lần**.

In [9]:
import time, random
for n_ in (16, 20, 24):
    coins = [0] * n_
    t = time.perf_counter()
    cheapest_brute(coins, 2)
    print(f"n={n_}, maxJump=2: {time.perf_counter() - t:.3f}s")

n=16, maxJump=2: 0.001s
n=20, maxJump=2: 0.004s
n=24, maxJump=2: 0.041s


## 6. Bước đột phá

**Insight 1 (bài toán con là hậu tố).** Đường tốt nhất từ ô `i` = `[i]` + đường tốt nhất từ ô `j` nào đó. Cả chi phí lẫn thứ tự từ điển đều "tách" được theo hậu tố:
- Chi phí: `coins[i] + chi phí từ j`.
- Từ điển: so phần tử thứ hai (`j`) trước; hòa `j` thì so tiếp hậu tố từ `j`.

**Insight 2 (chọn `j` nhỏ nhất khi hòa).** Trong các `j` có `dp[j]` nhỏ nhất, chọn `j` **nhỏ nhất**: đường đó nhỏ hơn mọi đường nhảy tới `j` lớn hơn, bất kể phần sau.

**Insight 3 (min trên cửa sổ).** `dp[i]` cần `min(dp[i+1 .. i+maxJump])` (ưu tiên chỉ số nhỏ khi hòa) → min trên cửa sổ trượt → deque đơn điệu, `O(n)` tổng.

**Invariant:** sau khi xử lý ô `i` (đi từ phải sang trái), `dp[i]` = chi phí nhỏ nhất từ `i` tới đích, và đường `i → nxt[i] → nxt[nxt[i]] → ...` là đường nhỏ nhất theo từ điển trong các đường chi phí nhỏ nhất từ `i`.

**Quy tắc nếu…thì…:**
- Nếu `coins[i] == -1` thì `dp[i] = ∞` (không bao giờ được chọn).
- Nếu `dp[j] + coins[i] < dp[i]` (**nhỏ hơn hẳn**) thì cập nhật — duyệt `j` tăng dần nên `j` đầu tiên đạt min được giữ.
- Nếu `dp[0] = ∞` thì trả `[]`.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Liệt kê mọi đường (mục 5) | hàm mũ | `O(n)` |
| 2 | Đệ quy có nhớ trả về `(chi phí, đường)` | `O(n · B · n)` | `O(n^2)` |
| 3 | DP ngược + `nxt` | `O(n · B)` | `O(n)` |
| 4 | DP ngược + deque đơn điệu | `O(n)` | `O(n)` |

(`B` = `maxJump`.)

### 7.1. Cách 2 — Đệ quy có nhớ trả về cả đường đi

**Giải thích.** `best(i)` trả về tuple `(chi phí, đường)` tốt nhất từ `i` tới đích, hoặc `None` nếu không tới được. Kết hợp: `(coins[i] + c, (i+1,) + path)` cho mỗi `j`, lấy `min`. Đường lưu dạng **tuple** để `lru_cache` dùng được và so sánh từ điển tự nhiên.

In [10]:
from functools import lru_cache

def cheapest_memo(coins, B):
    n = len(coins)

    @lru_cache(maxsize=None)
    def best(i):
        if coins[i] == -1:
            return None
        if i == n - 1:
            return (coins[i], (n,))
        res = None
        for j in range(i + 1, min(n, i + B + 1)):
            sub = best(j)
            if sub is not None:
                cand = (coins[i] + sub[0], (i + 1,) + sub[1])
                if res is None or cand < res:
                    res = cand
        return res

    r = best(0)
    return list(r[1]) if r else []

print(cheapest_memo(ex1, 2), cheapest_memo(ex1, 1), cheapest_memo([0, 0, 0, 0], 2))
assert cheapest_memo([0, 0, 0, 0], 2) == [1, 2, 3, 4]

[1, 3, 5] [] [1, 2, 3, 4]


**Đọc code:**

- `cand < res` so sánh tuple `(chi phí, đường)` — đúng luật đề, không cần nghĩ về "chọn `j` nhỏ nhất".
- Tạo `(i + 1,) + sub[1]` tốn `O(n)` → mỗi trạng thái `O(B · n)`.
- Đệ quy sâu tới `n = 1000` — vượt giới hạn mặc định (~1000) của Python; trên LeetCode cần `sys.setrecursionlimit`, hoặc tốt hơn là chuyển sang vòng lặp (Cách 3).

**Code nộp LeetCode (Cách 2)** — chuyển thành vòng lặp ngược để khỏi lo đệ quy, vẫn lưu cả đường:

In [11]:
from typing import List

class SolutionPaths:
    def cheapestJump(self, coins: List[int], maxJump: int) -> List[int]:
        n = len(coins)
        best = [None] * n                       # best[i] = (chi phí, tuple đường) hoặc None
        if coins[-1] != -1:
            best[-1] = (coins[-1], (n,))
        for i in range(n - 2, -1, -1):
            if coins[i] == -1:
                continue
            res = None
            for j in range(i + 1, min(n, i + maxJump + 1)):
                if best[j] is not None:
                    cand = (coins[i] + best[j][0], (i + 1,) + best[j][1])
                    if res is None or cand < res:
                        res = cand
            best[i] = res
        return list(best[0][1]) if best[0] else []

assert SolutionPaths().cheapestJump(ex1, 2) == [1, 3, 5]
assert SolutionPaths().cheapestJump(ex1, 1) == []
print("SolutionPaths OK")

SolutionPaths OK


**Độ phức tạp Cách 2:** Time `O(n · B · n)` (so sánh/ghép tuple độ dài `O(n)`), Space `O(n^2)`. Với `n = 1000, B = 100` là ~`10^8` — chậm.

### 7.2. Cách 3 — DP ngược + mảng `nxt`

**Giải thích.** Như mục 4: chỉ lưu `dp[i]` (số) và `nxt[i]`. Duyệt `j` **tăng dần**, cập nhật khi **nhỏ hơn hẳn** → `j` nhỏ nhất đạt min được giữ (Insight 2).

In [12]:
def cheapest_dp(coins, B):
    n = len(coins)
    INF = float('inf')
    dp = [INF] * n
    nxt = [-1] * n
    if coins[-1] != -1:
        dp[-1] = coins[-1]
    for i in range(n - 2, -1, -1):
        if coins[i] == -1:
            continue
        for j in range(i + 1, min(n, i + B + 1)):
            if coins[i] + dp[j] < dp[i]:
                dp[i] = coins[i] + dp[j]
                nxt[i] = j
    if dp[0] == INF:
        return []
    path, i = [], 0
    while i != -1:
        path.append(i + 1)
        i = nxt[i]
    return path

print(cheapest_dp(ex1, 2), cheapest_dp(ex1, 1), cheapest_dp([0, 0, 0, 0], 2))
assert cheapest_dp(ex1, 2) == [1, 3, 5]

[1, 3, 5] [] [1, 2, 3, 4]


**Đọc code:**

- `dp[j] = INF` cho ô `-1` hoặc ô không tới được đích → `coins[i] + INF = INF`, không bao giờ `< dp[i]`.
- Dựng đường: từ `0` đi theo `nxt` cho tới `-1` (ô cuối có `nxt = -1`). Chỉ ô cuối có `nxt = -1` trên đường này, vì mọi ô khác trên đường có `dp` hữu hạn nên đã có `nxt`.

**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionDP:
    def cheapestJump(self, coins: List[int], maxJump: int) -> List[int]:
        n = len(coins)
        INF = float('inf')
        dp = [INF] * n
        nxt = [-1] * n
        if coins[-1] != -1:
            dp[-1] = coins[-1]
        for i in range(n - 2, -1, -1):
            if coins[i] == -1:
                continue
            best, arg = INF, -1
            for j in range(i + 1, min(n, i + maxJump + 1)):
                if dp[j] < best:                 # "<" giữ j nhỏ nhất khi hòa
                    best, arg = dp[j], j
            if best < INF:
                dp[i], nxt[i] = coins[i] + best, arg
        if dp[0] == INF:
            return []
        res, i = [], 0
        while i != -1:
            res.append(i + 1)
            i = nxt[i]
        return res

assert SolutionDP().cheapestJump(ex1, 2) == [1, 3, 5]
assert SolutionDP().cheapestJump([0, 0, 0, 0], 2) == [1, 2, 3, 4]
print("SolutionDP OK")

SolutionDP OK


**Độ phức tạp Cách 3:** Time `O(n · B)` ≤ `10^5`, Space `O(n)`.

### 7.3. Cách 4 — DP ngược + deque đơn điệu (`O(n)`)

**Giải thích.** Khi `i` giảm dần, cửa sổ `[i+1, i+B]` trượt sang trái: chỉ số `i+1` **vào** (nhỏ nhất trong cửa sổ), chỉ số `i+B+1` **ra** (lớn nhất). Ta cần `min dp` trong cửa sổ, hòa thì chỉ số **nhỏ nhất**.

Chỉ số mới `i+1` **trội** hơn mọi chỉ số `j > i+1` có `dp[j] >= dp[i+1]`: nó tốt bằng hoặc hơn, nhỏ hơn (thắng khi hòa), và ở lại cửa sổ **lâu hơn** (vì các chỉ số lớn rời trước). Vậy:

```text
deque lưu chỉ số tăng dần từ trái sang phải, dp GIẢM dần nghiêm ngặt
vào: while dq và dp[dq[0]] >= dp[i+1]: popleft;  appendleft(i+1)
ra:  while dq[-1] > i + B: pop
min: dq[-1]
```

Ô có `dp = ∞` không cần đưa vào.

In [14]:
def cheapest_deque(coins, B):
    n = len(coins)
    INF = float('inf')
    dp = [INF] * n
    nxt = [-1] * n
    if coins[-1] != -1:
        dp[-1] = coins[-1]
    dq = deque()                          # chỉ số tăng dần (trái -> phải), dp giảm dần
    for i in range(n - 2, -1, -1):
        j_new = i + 1
        if dp[j_new] < INF:
            while dq and dp[dq[0]] >= dp[j_new]:
                dq.popleft()
            dq.appendleft(j_new)
        while dq and dq[-1] > i + B:
            dq.pop()
        if coins[i] != -1 and dq:
            j = dq[-1]
            dp[i], nxt[i] = coins[i] + dp[j], j
    if dp[0] == INF:
        return []
    path, i = [], 0
    while i != -1:
        path.append(i + 1)
        i = nxt[i]
    return path

print(cheapest_deque(ex1, 2), cheapest_deque(ex1, 1), cheapest_deque([0, 0, 0, 0], 2))
assert cheapest_deque(ex1, 2) == [1, 3, 5] and cheapest_deque([0, 0, 0, 0], 2) == [1, 2, 3, 4]

[1, 3, 5] [] [1, 2, 3, 4]


**Đọc code:**

- `dp[dq[0]] >= dp[j_new]` (có dấu **bằng**): khi hòa, chỉ số cũ (lớn hơn) bị loại → ưu tiên chỉ số nhỏ.
- Phần tử bên phải nhất `dq[-1]` có `dp` nhỏ nhất (deque giảm dần từ trái sang phải).
- Loại phần tử hết hạn ở **bên phải** (chỉ số lớn nhất) — ngược với sliding window xuôi thông thường.
- Ô `i` là `-1` vẫn phải cập nhật deque (vì cửa sổ vẫn trượt), chỉ là không tính `dp[i]`.

**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionDeque:
    def cheapestJump(self, coins: List[int], maxJump: int) -> List[int]:
        n = len(coins)
        INF = float('inf')
        dp = [INF] * n
        nxt = [-1] * n
        if coins[-1] != -1:
            dp[-1] = coins[-1]
        dq = deque()
        for i in range(n - 2, -1, -1):
            if dp[i + 1] < INF:
                while dq and dp[dq[0]] >= dp[i + 1]:
                    dq.popleft()
                dq.appendleft(i + 1)
            while dq and dq[-1] > i + maxJump:
                dq.pop()
            if coins[i] != -1 and dq:
                dp[i] = coins[i] + dp[dq[-1]]
                nxt[i] = dq[-1]
        if dp[0] == INF:
            return []
        res, i = [], 0
        while i != -1:
            res.append(i + 1)
            i = nxt[i]
        return res

assert SolutionDeque().cheapestJump(ex1, 2) == [1, 3, 5]
print("SolutionDeque OK")

SolutionDeque OK


**Độ phức tạp Cách 4:** mỗi chỉ số vào/ra deque tối đa một lần → Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước (Cách 4)

Trace `coins = [1, 2, 4, -1, 2]`, `maxJump = 2`:

| i | vào deque | deque sau khi vào | loại hết hạn (> i+2) | min (dq[-1]) | dp[i] | nxt[i] |
|---|---|---|---|---|---|---|
| 3 | 4 (dp=2) | [4] | — | 4 | ∞ (coin -1) | — |
| 2 | 3 (dp=∞) bỏ qua | [4] | — | 4 | 4+2 = 6 | 4 |
| 1 | 2 (dp=6) | [2, 4] | 4 > 3: bỏ | 2 | 2+6 = 8 | 2 |
| 0 | 1 (dp=8) | [1, 2] | — | 2 | 1+6 = 7 | 2 |

In [16]:
def trace_deque(coins, B):
    n = len(coins)
    INF = float('inf')
    dp, nxt = [INF] * n, [-1] * n
    if coins[-1] != -1:
        dp[-1] = coins[-1]
    dq = deque()
    print(f"dp[{n - 1}] = {dp[-1]} (đích)")
    for i in range(n - 2, -1, -1):
        j_new = i + 1
        note = []
        if dp[j_new] < INF:
            removed = []
            while dq and dp[dq[0]] >= dp[j_new]:
                removed.append(dq.popleft())
            dq.appendleft(j_new)
            note.append(f"vào {j_new} (dp={dp[j_new]})" + (f", loại trội {removed}" if removed else ""))
        else:
            note.append(f"{j_new} có dp=inf, bỏ qua")
        expired = []
        while dq and dq[-1] > i + B:
            expired.append(dq.pop())
        if expired:
            note.append(f"hết hạn {expired}")
        if coins[i] != -1 and dq:
            dp[i], nxt[i] = coins[i] + dp[dq[-1]], dq[-1]
        print(f"i={i}: {'; '.join(note)} | deque={list(dq)} | dp[{i}]={dp[i]}, nxt={nxt[i]}")
    path, i = [], 0
    if dp[0] < INF:
        while i != -1:
            path.append(i + 1)
            i = nxt[i]
    print("đường:", path, "| chi phí:", dp[0])
    return path

assert trace_deque(ex1, 2) == [1, 3, 5]

dp[4] = 2 (đích)
i=3: vào 4 (dp=2) | deque=[4] | dp[3]=inf, nxt=-1
i=2: 3 có dp=inf, bỏ qua | deque=[4] | dp[2]=6, nxt=4
i=1: vào 2 (dp=6); hết hạn [4] | deque=[2] | dp[1]=8, nxt=2
i=0: vào 1 (dp=8) | deque=[1, 2] | dp[0]=7, nxt=2
đường: [1, 3, 5] | chi phí: 7


Trace thêm một ví dụ có hòa chi phí để thấy quy tắc "hòa thì chỉ số nhỏ":

In [17]:
assert trace_deque([0, 0, 0, 0, 0], 3) == [1, 2, 3, 4, 5]

dp[4] = 0 (đích)
i=3: vào 4 (dp=0) | deque=[4] | dp[3]=0, nxt=4
i=2: vào 3 (dp=0), loại trội [4] | deque=[3] | dp[2]=0, nxt=3
i=1: vào 2 (dp=0), loại trội [3] | deque=[2] | dp[1]=0, nxt=2
i=0: vào 1 (dp=0), loại trội [2] | deque=[1] | dp[0]=0, nxt=1
đường: [1, 2, 3, 4, 5] | chi phí: 0


## 9. Code nộp LeetCode

Lời giải chính: **DP ngược + `nxt`** (Cách 3) — đơn giản, `O(n · maxJump)` đã đủ nhanh với ràng buộc.

In [18]:
from typing import List

class Solution:
    def cheapestJump(self, coins: List[int], maxJump: int) -> List[int]:
        n = len(coins)
        INF = float('inf')
        dp = [INF] * n        # dp[i]: chi phí nhỏ nhất từ ô i tới ô cuối (tính cả hai đầu)
        nxt = [-1] * n        # ô kế tiếp trên đường tốt nhất
        if coins[-1] != -1:
            dp[-1] = coins[-1]
        for i in range(n - 2, -1, -1):
            if coins[i] == -1:
                continue
            best, arg = INF, -1
            for j in range(i + 1, min(n, i + maxJump + 1)):
                if dp[j] < best:          # nhỏ hơn HẲN: hòa thì giữ j nhỏ nhất
                    best, arg = dp[j], j
            if best < INF:
                dp[i] = coins[i] + best
                nxt[i] = arg
        if dp[0] == INF:
            return []
        path, i = [], 0
        while i != -1:
            path.append(i + 1)        # đổi sang chỉ số 1-based
            i = nxt[i]
        return path

sol = Solution()
assert sol.cheapestJump([1, 2, 4, -1, 2], 2) == [1, 3, 5]
assert sol.cheapestJump([1, 2, 4, -1, 2], 1) == []
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

Gọi `Best(i)` là đường tốt nhất từ `i` tới đích theo thứ tự `(chi phí, đường)` (chi phí nhỏ nhất, hòa thì từ điển nhỏ nhất).

**Bổ đề (cấu trúc con tối ưu theo cả hai tiêu chí).** Nếu `Best(i) = [i, j, ...]` thì phần `[j, ...]` chính là `Best(j)`.

*Chứng minh.* Nếu `[j, ...]` không phải `Best(j)`, thay nó bằng `Best(j)`: chi phí giảm (mâu thuẫn tối ưu chi phí) hoặc bằng chi phí nhưng nhỏ hơn theo từ điển — khi đó `[i] + Best(j)` cũng nhỏ hơn `[i, j, ...]` theo từ điển (cùng hai phần tử đầu `i, j`, phần sau nhỏ hơn) — mâu thuẫn.

**Chọn `j`.** `Best(i) = [i] + Best(j*)` với `j*` tối ưu theo `(coins[i] + dp[j], [i] + Best(j))`. Trong các `j` có `dp[j]` nhỏ nhất (cùng chi phí tổng), so sánh từ điển: phần tử đầu `i` như nhau, phần tử thứ hai là `j` → **`j` nhỏ nhất thắng**, không cần xem phần sau. Code duyệt `j` tăng dần và chỉ cập nhật khi **nhỏ hơn hẳn** → chọn đúng `j` nhỏ nhất đạt min.

**Quy nạp** từ `i = n-1` về `0` cho `dp[i]` và `nxt[i]` đúng; dựng đường theo `nxt` cho `Best(0)`.

**Deque (Cách 4)** trả về đúng `j` nhỏ nhất có `dp[j]` nhỏ nhất trong cửa sổ: một chỉ số chỉ bị loại khi có chỉ số **nhỏ hơn** với `dp` **không lớn hơn** — nó không bao giờ có thể là đáp án duy nhất tốt nhất về sau (chỉ số nhỏ hơn còn ở lại cửa sổ lâu hơn).

**Vì sao DP xuôi sai?** DP xuôi nhớ `prev[j]` (ô đứng trước `j`). Thứ tự từ điển quyết định bởi các phần tử **đầu** của đường, nhưng `prev` chỉ quyết định phần **cuối** — chọn `prev` nhỏ nhất không đảm bảo phần đầu nhỏ nhất (xem Lỗi 1).

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute | hàm mũ | `O(n)` |
| 2. Memo đường | `O(n^2 · B)` | `O(n^2)` |
| 3. DP + nxt | `O(n · B)` | `O(n)` |
| 4. Deque | `O(n)` | `O(n)` |

In [19]:
random.seed(7)
big = [random.choice([-1] + list(range(0, 101)) * 3) for _ in range(1000)]
big[0] = 5
big[-1] = 3
for name, f in [("paths", SolutionPaths().cheapestJump), ("dp", SolutionDP().cheapestJump), ("deque", SolutionDeque().cheapestJump)]:
    t = time.perf_counter()
    r = f(big, 100)
    print(f"{name:>6}: độ dài đường {len(r)}, chi phí {path_cost(big, r)}  {time.perf_counter() - t:.3f}s")
assert SolutionDP().cheapestJump(big, 100) == SolutionDeque().cheapestJump(big, 100) == SolutionPaths().cheapestJump(big, 100)

 paths: độ dài đường 21, chi phí 11  0.012s
    dp: độ dài đường 21, chi phí 11  0.002s
 deque: độ dài đường 21, chi phí 11  0.000s


## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1`, ô cuối `-1`, hòa chi phí → từ điển, `maxJump` lớn hơn `n`, nhiều `-1` liên tiếp) và đối chiếu ngẫu nhiên có seed giữa mọi cách với brute force.

In [20]:
def all_methods(coins, B):
    return {
        "brute": cheapest_brute(coins, B),
        "memo": cheapest_memo(coins, B),
        "dp": cheapest_dp(coins, B),
        "deque": cheapest_deque(coins, B),
        "SolutionPaths": SolutionPaths().cheapestJump(coins, B),
        "SolutionDP": SolutionDP().cheapestJump(coins, B),
        "SolutionDeque": SolutionDeque().cheapestJump(coins, B),
        "Solution": Solution().cheapestJump(coins, B),
    }

cases = [
    (([1, 2, 4, -1, 2], 2), [1, 3, 5]),
    (([1, 2, 4, -1, 2], 1), []),
    (([7], 1), [1]),                                  # n = 1
    (([1, 2, -1], 5), []),                            # ô cuối -1
    (([0, 0, 0, 0], 2), [1, 2, 3, 4]),                # hòa: đường dài hơn nhưng nhỏ hơn theo từ điển
    (([1, 1, 1, 1], 3), [1, 4]),
    (([0, -1, -1, 0], 2), []),                        # hai -1 liên tiếp chặn đường
    (([0, -1, -1, 0], 3), [1, 4]),
    (([2, 2, 2, 0, 0], 2), [1, 2, 4, 5]),
]
for (coins, B), want in cases:
    for name, got in all_methods(coins, B).items():
        assert got == want, (coins, B, name, got, want)
print("Tất cả", len(cases), "test cố định OK")

Tất cả 9 test cố định OK


In [21]:
random.seed(2024)
for trial in range(1500):
    n_ = random.randint(1, 9)
    coins = [random.choice([-1, 0, 0, 1, 1, 2, 3]) for _ in range(n_)]
    if coins[0] == -1:
        coins[0] = random.randint(0, 3)
    B = random.randint(1, 4)
    res = all_methods(coins, B)
    want = res["brute"]
    for name, got in res.items():
        assert got == want, (coins, B, name, got, want)
print("1500 test ngẫu nhiên: mọi cách khớp brute force")

1500 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: DP xuôi với `prev` nhỏ nhất

**Code sai:** `dp[j]` = chi phí nhỏ nhất từ đầu tới `j`, `prev[j]` = ô trước nhỏ nhất đạt min; dựng ngược từ cuối.

**Vì sao sai:** thứ tự từ điển phụ thuộc phần **đầu** đường, còn `prev` chỉ tối ưu phần cuối.

In [22]:
def forward_wrong(coins, B):
    n = len(coins)
    INF = float('inf')
    dp, prev = [INF] * n, [-1] * n
    dp[0] = coins[0]
    for j in range(1, n):
        if coins[j] == -1:
            continue
        for i in range(max(0, j - B), j):
            if dp[i] + coins[j] < dp[j]:
                dp[j], prev[j] = dp[i] + coins[j], i
    if dp[-1] == INF:
        return []
    path, j = [], n - 1
    while j != -1:
        path.append(j + 1)
        j = prev[j]
    return path[::-1]

for coins, B in [([0, 0, 0, 0], 2), ([2, 2, 2, 0, 0], 2)]:
    print(coins, B, "sai:", forward_wrong(coins, B), "| đúng:", Solution().cheapestJump(coins, B))

[0, 0, 0, 0] 2 sai: [1, 2, 4] | đúng: [1, 2, 3, 4]
[2, 2, 2, 0, 0] 2 sai: [1, 3, 5] | đúng: [1, 2, 4, 5]


**Code đúng:** DP **ngược** với `nxt` (Cách 3).

### Lỗi 2: Dùng `<=` khi chọn `j` → chọn `j` lớn nhất khi hòa

**Code sai:** `if dp[j] <= best` với `j` tăng dần → `j` cuối cùng đạt min được giữ.

In [23]:
def dp_le_wrong(coins, B):
    n = len(coins)
    INF = float('inf')
    dp, nxt = [INF] * n, [-1] * n
    if coins[-1] != -1:
        dp[-1] = coins[-1]
    for i in range(n - 2, -1, -1):
        if coins[i] == -1:
            continue
        best, arg = INF, -1
        for j in range(i + 1, min(n, i + B + 1)):
            if dp[j] <= best and dp[j] < INF:      # SAI: <=
                best, arg = dp[j], j
        if arg != -1:
            dp[i], nxt[i] = coins[i] + best, arg
    if dp[0] == INF:
        return []
    path, i = [], 0
    while i != -1:
        path.append(i + 1)
        i = nxt[i]
    return path

print("sai :", dp_le_wrong([0, 0, 0, 0], 2), "| đúng:", Solution().cheapestJump([0, 0, 0, 0], 2))

sai : [1, 3, 4] | đúng: [1, 2, 3, 4]


**Code đúng:** `<` nghiêm ngặt khi duyệt `j` tăng dần.

### Lỗi 3: Quên chi phí ô đầu / ô cuối, hoặc lệch 0/1-based

**Code sai:** trả về chỉ số 0-based, hoặc `dp[n-1] = 0`. Chi phí ô đầu/cuối không ảnh hưởng **đường** được chọn (mọi đường đều qua hai ô đó), nhưng trả chỉ số 0-based thì sai ngay.

In [24]:
def zero_based_wrong(coins, B):
    return [i - 1 for i in Solution().cheapestJump(coins, B)]      # SAI: đề dùng 1-based
print("sai :", zero_based_wrong(ex1, 2), "| đúng:", Solution().cheapestJump(ex1, 2))

sai : [0, 2, 4] | đúng: [1, 3, 5]


**Code đúng:** `path.append(i + 1)` khi dựng đường.

### Lỗi 4: Cho rằng đường ít bước hơn luôn thắng khi hòa chi phí

Như mục 4.4: `[1, 2, 3, 4] < [1, 2, 4]` theo từ điển dù dài hơn. Đừng thêm tiêu chí "ít bước" — chỉ có chi phí rồi từ điển.

## 14. Mẹo & nhận dạng pattern

- **"Đường đi tối ưu + hòa thì nhỏ nhất theo từ điển"** → DP **từ đích về nguồn**, khi hòa chọn **bước kế tiếp nhỏ nhất**. (Cùng ý tưởng với nhiều bài BFS/DP yêu cầu đường từ điển nhỏ nhất.)
- `dp[i]` phụ thuộc `min` của `B` ô kế tiếp → cửa sổ trượt → deque đơn điệu.
- Truy vết: lưu `nxt`/`prev` thay vì cả đường.
- Ô cấm (`-1`) → gán `∞`, tự động không được chọn.
- Bài cùng họ: 55/45 (Jump Game I/II), 1696 (Jump Game VI — deque đơn điệu), 746 (Min Cost Climbing Stairs), 1871 (Jump Game VII).

## 15. Biến thể

1. **Chỉ cần chi phí** (không cần đường) — DP xuôi hay ngược đều được; deque cho `O(n)`.
2. **Hòa thì đường lớn nhất theo từ điển** — chọn `j` **lớn nhất** khi hòa (dùng `<=`).
3. **Jump Game VI (LC 1696)** — tối đa hóa tổng, không có ô cấm: deque đơn điệu xuôi.
4. **Bước nhảy có chi phí riêng** — `dp[i] = coins[i] + min(cost(i,j) + dp[j])`.

In [25]:
def cheapest_lex_largest(coins, B):
    """Biến thể 2: hòa chi phí thì đường LỚN nhất theo từ điển."""
    n = len(coins)
    INF = float('inf')
    dp, nxt = [INF] * n, [-1] * n
    if coins[-1] != -1:
        dp[-1] = coins[-1]
    for i in range(n - 2, -1, -1):
        if coins[i] == -1:
            continue
        for j in range(i + 1, min(n, i + B + 1)):
            if dp[j] < INF and coins[i] + dp[j] <= dp[i]:     # <=: j lớn nhất thắng
                dp[i], nxt[i] = coins[i] + dp[j], j
    if dp[0] == INF:
        return []
    path, i = [], 0
    while i != -1:
        path.append(i + 1)
        i = nxt[i]
    return path

def brute_lex_largest(coins, B):
    n = len(coins)
    paths = []
    def dfs(i, p, c):
        if i == n - 1:
            paths.append((c, p))
            return
        for j in range(i + 1, min(n, i + B + 1)):
            if coins[j] != -1:
                dfs(j, p + [j + 1], c + coins[j])
    if coins[0] != -1:
        dfs(0, [1], coins[0])
    if not paths:
        return []
    mc = min(c for c, _ in paths)
    return max(p for c, p in paths if c == mc)

random.seed(5)
for _ in range(500):
    coins = [random.choice([-1, 0, 1, 2]) for _ in range(random.randint(1, 8))]
    coins[0] = max(coins[0], 0)
    B = random.randint(1, 3)
    assert cheapest_lex_largest(coins, B) == brute_lex_largest(coins, B)
print("Biến thể 2 OK:", cheapest_lex_largest([0, 0, 0, 0], 2))

Biến thể 2 OK: [1, 3, 4]


In [26]:
def max_result_jump_game_vi(nums, k):
    """Biến thể 3 (LC 1696): tối đa tổng khi nhảy tối đa k bước, deque đơn điệu xuôi."""
    n = len(nums)
    dp = [0] * n
    dp[0] = nums[0]
    dq = deque([0])
    for i in range(1, n):
        while dq[0] < i - k:
            dq.popleft()
        dp[i] = nums[i] + dp[dq[0]]
        while dq and dp[dq[-1]] <= dp[i]:
            dq.pop()
        dq.append(i)
    return dp[-1]

print(max_result_jump_game_vi([1, -1, -2, 4, -7, 3], 2), max_result_jump_game_vi([10, -5, -2, 4, 0, 3], 3))
assert max_result_jump_game_vi([1, -1, -2, 4, -7, 3], 2) == 7

7 17


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | liệt kê mọi đường, min `(chi phí, đường)` | hàm mũ | `O(n)` | trọng tài, `n` ≤ 20 |
| 2. Memo đường | lưu cả đường tốt nhất mỗi ô | `O(n^2 B)` | `O(n^2)` | khi chưa nghĩ ra quy tắc chọn `j` |
| 3. DP + nxt | DP ngược, hòa chọn `j` nhỏ nhất | `O(nB)` | `O(n)` | bản nộp chuẩn |
| 4. Deque | như 3 + min cửa sổ trượt | `O(n)` | `O(n)` | `B` lớn |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `cheapestJump([0, 5, 0, 5, 0], 2)`, `cheapestJump([3, 1, 1, 3], 3)`, `cheapestJump([1, -1, 1], 1)`?

**Bài 2 (tính tay).** Lập bảng DP ngược (như mục 4.3) cho `coins = [2, 1, 3, 1, 2, 1]`, `maxJump = 3`.

**Bài 3 (điền TODO).** Hoàn thiện `build_path(nxt)` dựng đường 1-based từ mảng `nxt`.

**Bài 4 (tự viết lại).** Viết lại Cách 3 `my_cheapest_jump(coins, B)`.

**Bài 5 (mini challenge).** Viết `count_cheapest_paths(coins, B)` đếm **số đường** đạt chi phí nhỏ nhất (DP ngược, đếm song song với `dp`).

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def build_path(nxt):
    # TODO: i = 0; while i != -1: append(i + 1); i = nxt[i]
    pass

# Bài 4 — TODO
def my_cheapest_jump(coins, B):
    # TODO: dp ngược, j tăng dần, cập nhật khi nhỏ hơn hẳn
    pass

# Bài 5 — TODO
def count_cheapest_paths(coins, B):
    # TODO: ways[i] = tổng ways[j] với các j đạt min dp
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: ô có `0` là rẻ nhất, cố đứng trên chúng.
- Mức 2: `[0,5,0,5,0]`, B = 2: `1 → 3 → 5` chi phí 0.
- Mức 3: `[3,1,1,3]`, B = 3: `[1,4]` chi phí 6, `[1,2,4]` = 7 … → `[1, 4]`; `[1,-1,1]`, B = 1 → `[]`.

**Bài 2.**
- Mức 1: `dp[5] = 1`.
- Mức 2: `dp[4] = 3`, `dp[3] = 1 + min(3, 1) = 2`, `dp[2] = 3 + min(2, 3, 1) = 4`.
- Mức 3: `dp[1] = 1 + min(4, 2, 3) = 3`, `dp[0] = 2 + min(3, 4, 2) = 4`; min đạt tại `j = 3` nên `nxt[0] = 3`, tiếp theo `nxt[3] = 5` → đường `[1, 4, 6]`.

**Bài 3.**
- Mức 1: `path = []`.
- Mức 2: vòng `while i != -1`.
- Mức 3: `append(i + 1)` rồi `i = nxt[i]`.

**Bài 4.**
- Mức 1: `dp[-1] = coins[-1]` nếu khác `-1`.
- Mức 2: `i` từ `n-2` về `0`; bỏ qua ô `-1`.
- Mức 3: `if dp[j] < best`.

**Bài 5.**
- Mức 1: `ways[n-1] = 1`.
- Mức 2: với mỗi `i`, tìm `best = min dp[j]`, `ways[i] = Σ ways[j]` với `dp[j] == best`.
- Mức 3: kiểm tra bằng brute force liệt kê đường.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert Solution().cheapestJump([0, 5, 0, 5, 0], 2) == [1, 3, 5]
assert Solution().cheapestJump([3, 1, 1, 3], 3) == [1, 4]
assert Solution().cheapestJump([1, -1, 1], 1) == []

# Bài 2
backward_table([2, 1, 3, 1, 2, 1], 3)
assert Solution().cheapestJump([2, 1, 3, 1, 2, 1], 3) == [1, 4, 6]

# Bài 3
def build_path_ans(nxt):
    path, i = [], 0
    while i != -1:
        path.append(i + 1)
        i = nxt[i]
    return path

assert build_path_ans([2, -1, 4, -1, -1]) == [1, 3, 5]
print("Bài 1–3 OK")

 i | coin | ứng viên (j: dp[j])          | dp[i] | nxt
 5 |    1 | (đích)                       |     1 | -1
 4 |    2 | 5: 1                         |     3 | 5
 3 |    1 | 4: 3, 5: 1                   |     2 | 5
 2 |    3 | 3: 2, 4: 3, 5: 1             |     4 | 5
 1 |    1 | 2: 4, 3: 2, 4: 3             |     3 | 3
 0 |    2 | 1: 3, 2: 4, 3: 2             |     4 | 3
Bài 1–3 OK


In [29]:
# Bài 4
def my_cheapest_jump_ans(coins, B):
    n = len(coins)
    INF = float('inf')
    dp, nxt = [INF] * n, [-1] * n
    if coins[-1] != -1:
        dp[-1] = coins[-1]
    for i in range(n - 2, -1, -1):
        if coins[i] == -1:
            continue
        for j in range(i + 1, min(n, i + B + 1)):
            if coins[i] + dp[j] < dp[i]:
                dp[i], nxt[i] = coins[i] + dp[j], j
    return build_path_ans(nxt) if dp[0] < INF else []

# Bài 5
def count_cheapest_paths_ans(coins, B):
    n = len(coins)
    INF = float('inf')
    dp, ways = [INF] * n, [0] * n
    if coins[-1] != -1:
        dp[-1], ways[-1] = coins[-1], 1
    for i in range(n - 2, -1, -1):
        if coins[i] == -1:
            continue
        window = range(i + 1, min(n, i + B + 1))
        best = min((dp[j] for j in window), default=INF)
        if best < INF:
            dp[i] = coins[i] + best
            ways[i] = sum(ways[j] for j in window if dp[j] == best)
    return ways[0]

def count_brute(coins, B):
    n = len(coins)
    costs = []
    def dfs(i, c):
        if i == n - 1:
            costs.append(c)
            return
        for j in range(i + 1, min(n, i + B + 1)):
            if coins[j] != -1:
                dfs(j, c + coins[j])
    if coins[0] != -1:
        dfs(0, coins[0])
    return costs.count(min(costs)) if costs else 0

random.seed(21)
for _ in range(800):
    coins = [random.choice([-1, 0, 0, 1, 2]) for _ in range(random.randint(1, 9))]
    coins[0] = max(coins[0], 0)
    B = random.randint(1, 4)
    assert my_cheapest_jump_ans(coins, B) == cheapest_brute(coins, B)
    assert count_cheapest_paths_ans(coins, B) == count_brute(coins, B)
print("Bài 4–5 OK (đối chiếu brute force); số đường rẻ nhất của [0,0,0,0], B=2:", count_cheapest_paths_ans([0, 0, 0, 0], 2))

Bài 4–5 OK (đối chiếu brute force); số đường rẻ nhất của [0,0,0,0], B=2: 3


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đường tối ưu, hòa thì nhỏ nhất theo từ điển":**

```python
dp = [INF] * n; nxt = [-1] * n
dp[n-1] = cost(n-1)
for i in range(n - 2, -1, -1):          # NGƯỢC: từ đích về nguồn
    if blocked(i): continue
    for j in next_choices(i):           # duyệt j TĂNG dần
        if cost(i) + dp[j] < dp[i]:     # nhỏ hơn HẲN
            dp[i], nxt[i] = cost(i) + dp[j], j
path = follow nxt from 0                # nhớ đổi sang 1-based
```

**Checklist:**

- [ ] DP **ngược** (từ ô cuối), không phải xuôi.
- [ ] Duyệt `j` tăng dần, cập nhật bằng `<` (giữ `j` nhỏ nhất khi hòa).
- [ ] Ô `-1` → `∞`; ô cuối `-1` → `[]`.
- [ ] Kết quả **1-based**.
- [ ] `n = 1` → `[1]`.
- [ ] Không thêm tiêu chí "ít bước hơn".